# Fixed-pose scoring and additive evaluations

This executed software control uses public DockingMT APIs and the previously retained minimal Vina input bytes. It evaluates the supplied conformation without search or optimization. The deliberately overlapping geometry is not a scientific docking benchmark. See [the contract](fixed_pose_scoring.md) and [issue #34](https://github.com/uibcdf/dockingmt/issues/34). No Meeko or MolSysViewer import is required.


In [1]:
import base64
import json
import platform
import sys
from pathlib import Path
import numpy as np
import pyunitwizard as puw
import molsysmt as msm
import vina
import dockingmt as dmt

assert sys.version_info[:2] == (3, 14)
assert 'molsyssuite@uibcdf_3.14' in sys.executable
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'dockingmt').is_dir())
manifest = json.loads((root / 'devguide/validation/data/audit/minimal_vina_result.json').read_text())
artifacts = manifest['provenance']['backend_artifacts']
problem = dmt.DockingProblem(
    receptor=base64.b64decode(artifacts['receptor']['content_base64']).decode(),
    partner=base64.b64decode(artifacts['partner']['content_base64']).decode(),
    search_domain=dmt.BoxRegion.from_dict(manifest['problem_info']['search_domain']),
)
print(json.dumps({'python': platform.python_version(), 'interpreter': sys.executable,
                 'dockingmt': dmt.__version__, 'molsysmt': msm.__version__,
                 'vina': vina.__version__, 'molsysmt_source': str(Path(msm.__file__).parent)}, indent=2))


{
  "python": "3.14.7",
  "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "dockingmt": "0.0.0+101.gbe39d74.dirty",
  "molsysmt": "0.22.4+24.g42b487869",
  "vina": "1.2.7",
  "molsysmt_source": "/home/diego/repos@uibcdf/dockingmt/.pytest_cache/codex-source-archives/dockingmt-template-provider-c19a47ada0c2279029abfa296cf915560610ad9a/molsysmt"
}


In [2]:
first = dmt.score(problem, dmt.VinaProtocol(cpu=1, seed=17, capture_backend_inputs=True), score_name='vina')
before = first.to_dict()
second = dmt.score(problem, dmt.VinaProtocol(cpu=1, seed=17, scoring='vinardo'), pose=first, score_name='vinardo')
assert first.to_dict() == before
np.testing.assert_array_equal(puw.get_value(first.coordinates), puw.get_value(second.coordinates))
assert len(second.scores) == 16
assert second.rank is None
for name, value in second.scores.items():
    definition = second.score_definitions[name]
    print(f"{name:30s} {value:10.3f} {definition['unit']} ({definition['kind']})")


vina                               38.197 kilocalorie / mole (empirical)
vina.lig_inter                     38.197 kilocalorie / mole (empirical)
vina.flex_inter                     0.000 kilocalorie / mole (empirical)
vina.other_inter                    0.000 kilocalorie / mole (empirical)
vina.flex_intra                     0.000 kilocalorie / mole (empirical)
vina.lig_intra                      0.000 kilocalorie / mole (empirical)
vina.torsions                       0.000 kilocalorie / mole (empirical)
vina.lig_intra_best_pose            0.000 kilocalorie / mole (empirical)
vinardo                            39.483 kilocalorie / mole (empirical)
vinardo.lig_inter                  39.483 kilocalorie / mole (empirical)
vinardo.flex_inter                  0.000 kilocalorie / mole (empirical)
vinardo.other_inter                 0.000 kilocalorie / mole (empirical)
vinardo.flex_intra                  0.000 kilocalorie / mole (empirical)
vinardo.lig_intra                   0.000 kilocalor

In [3]:
for evaluation in second.metadata['scoring_history']:
    report = {key: evaluation[key] for key in ('operation', 'backend_version', 'applied_parameters', 'unused_search_parameters', 'geometry_check')}
    report['input_hashes'] = {role: record['sha256'] for role, record in evaluation['backend_artifacts'].items()}
    report['preparation_assessment'] = {role: record['assessment_report']['assessment'] for role, record in evaluation['preparation'].items()}
    print(json.dumps(report, indent=2))
assert 'molsysviewer' not in sys.modules
assert 'meeko' not in sys.modules


{
  "operation": "score",
  "backend_version": "1.2.7",
  "applied_parameters": {
    "scoring": "vina",
    "cpu": 1,
    "seed": 17,
    "allow_provisional_preparation": false,
    "capture_backend_inputs": true,
    "collect_timings": false
  },
  "unused_search_parameters": [
    "exhaustiveness",
    "n_poses",
    "energy_range"
  ],
  "geometry_check": {
    "atom_order": "pdbqt_record_order",
    "identity_check": "positional_only",
    "coordinate_tolerance": {
      "value": 0.000500001,
      "unit": "angstrom"
    },
    "state_checks": {
      "partner": "unknown",
      "receptor": "unknown"
    }
  },
  "input_hashes": {
    "receptor": "fdf890083d54bf740b9ca1eab96069840bffbbf6cea4a8dfdbfa9651ba79d80b",
    "partner": "e195e663227eb6b77c2fe7b71efbc906ef6b5e72860261f0c18b8b37a43bf884"
  },
  "preparation_assessment": {
    "receptor": "unassessed",
    "partner": "unassessed"
  }
}
{
  "operation": "score",
  "backend_version": "1.2.7",
  "applied_parameters": {
    "scor

In [4]:
annotated = second.with_scores({'manual_review': 0.75})
assert 'manual_review' not in second.scores
assert 'manual_review' not in annotated.score_definitions  # Meaning remains undeclared.
restored = dmt.DockingPose.from_dict(json.loads(json.dumps(annotated.to_dict(), allow_nan=False)))
assert restored.to_dict() == annotated.to_dict()
try:
    second.with_scores({'vina': 0.0})
except Exception as error:
    print(type(error).__name__ + ': an existing score cannot be replaced')
else:
    raise AssertionError('Existing scores were overwritten')
print('Independent roundtrip:', len(restored.scores), 'scores;', len(restored.metadata['scoring_history']), 'native evaluations')


ArgumentError: an existing score cannot be replaced
Independent roundtrip: 17 scores; 2 native evaluations
